# 🕵️ Case file 01 — The Gold bag leak

> **Mission briefing.** It is 09:14 at Skyline Travel and the support queue is on fire. Basic-fare customers are being quoted Gold-fare baggage rules. As the Observability Detective on duty, your job is to open the activity logs and find out where the answer went wrong.

The assistant searches a policy database before answering — in technical terms, retrieval-augmented generation (RAG). Run the cells from top to bottom in a new kernel.

## Case objective
See how the search scope changes the answer: compare a search across all policies with a search limited to the customer's fare.

## What to expect
The unfiltered run retrieves the Gold policy and quotes it to a Basic-fare customer. The filtered run retrieves the Basic policy. Both runs leave an activity log: `activity-1-rag-no-filter` and `activity-1-rag-filter`.

## Bad practice
Search all policies and use the closest match, without checking whether the policy applies to the customer's fare.

## Good practice
Limit the search to policies for the customer's fare before choosing the closest match.

## Why the good practice works
The assistant cannot tell which fare the customer has unless the search is limited. Without the limit, a policy from a different fare can win the ranking and be quoted as if it applied.

## Deliverable
The two activity-log links and a short comparison of the answers and the policy each run found.


In [1]:
from IPython.display import Markdown, display
from experiment_utils import RUN_ID, langfuse, run_rag


## 🔮 Your prediction

The customer is on a **Basic** fare and asks how many checked bags they get.

**Which policy do you expect the unfiltered search to return?** And what will the assistant tell the customer?

Write your guess in a comment in the next cell, or double-click this cell and type it below. Commit to it before you run the experiment.

---
*Your guess:*


In [2]:
customer = "How many checked bags do I get on my booking?"
runs = [run_rag(customer, None, "no-filter"), run_rag(customer, "basic", "filter")]

for run in runs:
    if run["doc"]["tier"] != "basic":
        print(f"⚠️  Policy leak: {run['label']} returned a {run['doc']['tier']} policy for a Basic-fare customer.")
    else:
        print(f"✅  Correct policy: {run['label']} returned the {run['doc']['tier']} policy.")

rows = ["| search scope | policy found | applies to Basic fare? | answer | input tokens | output tokens | time (s) | activity log |",
        "|---|---|---|---|---|---|---|---|"]
for run in runs:
    doc = run["doc"]
    applies = "yes" if doc["tier"] == "basic" else "**no — leak**"
    answer = (run["answer"] or "").replace("\n", " ").replace("|", "\\|")
    rows.append(f"| {run['label']} | {doc['title']} ({doc['tier']}) | {applies} | {answer} | {run['input_tokens']} | {run['output_tokens']} | {run['latency']:.1f} | [Open log]({run['trace_url']}) |")
display(Markdown("\n".join(rows)))

leaked = [run for run in runs if run["doc"]["tier"] != "basic"]
fixed = [run for run in runs if run["doc"]["tier"] == "basic"]
notes = []
if leaked:
    notes.append(f"**What went wrong:** the {leaked[0]['label']} search returned *{leaked[0]['doc']['title']}* — a {leaked[0]['doc']['tier']}-fare policy — for a Basic-fare customer.")
if fixed:
    notes.append(f"**What fixed it:** limiting the search to Basic-fare policies returned *{fixed[0]['doc']['title']}*.")
display(Markdown("  \n".join(notes)))


⚠️  Policy leak: all policies returned a gold policy for a Basic-fare customer.
✅  Correct policy: Basic fare only returned the basic policy.


| search scope | policy found | applies to Basic fare? | answer | input tokens | output tokens | time (s) | activity log |
|---|---|---|---|---|---|---|---|
| all policies | Gold fare baggage (gold) | **no — leak** | You get **one 23 kg checked bag** included with the Gold fare. A second checked bag costs **60 EUR**. | 72 | 97 | 4.1 | [Open log](http://localhost:3000/project/langfuse-observability/traces/bcf49468bd029d8912956cf1c04b4e6e) |
| Basic fare only | Basic fare baggage (basic) | yes | You don’t get any checked bags included with the Basic fare. You can add a 23 kg checked bag for 45 EUR. | 82 | 103 | 2.0 | [Open log](http://localhost:3000/project/langfuse-observability/traces/65a0571ac00817d1334fa8abf98b5ab6) |

**What went wrong:** the all policies search returned *Gold fare baggage* — a gold-fare policy — for a Basic-fare customer.  
**What fixed it:** limiting the search to Basic-fare policies returned *Basic fare baggage*.

## 🔍 Reading the evidence

Open the unfiltered activity log and inspect the retrieval step. It shows the query went out with no fare filter, and the Gold document won the ranking. Then open the filtered log: the retrieval step records a `basic` filter and only Basic documents were searched.

Both answers sound confident. The difference is in the evidence the assistant was allowed to see — and that is exactly what the activity log makes visible.


## 🏁 Wrap-up

Run the final cell to close the case, show the activity-log links, and flush the traces to Langfuse. Submit the links and your findings.


In [3]:
lines = [f"**Session:** `activity-1-{RUN_ID}`", "", "**Activity logs:**", ""]
for run in runs:
    lines.append(f"- [{run['label']}]({run['trace_url']})")
display(Markdown("\n".join(lines)))
langfuse.flush()
print("✅ Incident closed: the search is now limited to the customer's fare.")
print("🕵️ Detective score: 1/4 incidents resolved. Next case: the rejected booking request (Activity 2).")
display(Markdown("Activity logs sent to Langfuse. Open them and compare the retrieval steps."))


**Session:** `activity-1-d939a73b`

**Activity logs:**

- [all policies](http://localhost:3000/project/langfuse-observability/traces/bcf49468bd029d8912956cf1c04b4e6e)
- [Basic fare only](http://localhost:3000/project/langfuse-observability/traces/65a0571ac00817d1334fa8abf98b5ab6)

✅ Incident closed: the search is now limited to the customer's fare.
🕵️ Detective score: 1/4 incidents resolved. Next case: the rejected booking request (Activity 2).


Activity logs sent to Langfuse. Open them and compare the retrieval steps.

## 🎁 Bonus — What else would you try?

- Add a topic filter (baggage, cancellation, seating) so unrelated policies never compete.
- Run the same query for a Silver-fare customer and check which policy wins.
- Look at the retrieval step's score in the trace: would a score threshold have caught this leak?
